# NBA Player Projection Query Tool

Query over/under probabilities, run Monte Carlo simulations, and compare betting lines using cached projections from the blended CatBoost + Transformer pipeline.

This notebook reads the projection output generated by `simulate_season.py`, so you do not need to rerun training or data collection here.

**Core features:**
- Over/under probability calculation (Normal CDF)
- Monte Carlo simulation (5,000+ iterations)
- Multi-line comparison tables
- Injury-adjusted probabilities
- Detailed formatted output with recommendations

In [1]:
# Setup — import the probability engine
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from src.query.probability_calculator import ProbabilityCalculator, ProbabilityResult

calc = ProbabilityCalculator()
print('ProbabilityCalculator ready.')

ProbabilityCalculator ready.


## Quick Query

Change `player`, `stat`, `line`, `mean`, `std` to query any player.

In [2]:
# --- EDIT THESE VALUES ---
player   = 'LeBron James'
team     = 'LAL'
opponent = 'BOS'
stat     = 'pts'       # pts, reb, ast, stl, blk, tov
line     = 25.5
mean     = 27.2        # projected average
std      = 6.8         # projected std dev
# -------------------------

result = calc.calculate_from_projection(
    player_name=player, stat=stat, line=line,
    mean=mean, std=std, opponent=opponent
)
result.team = team
print(calc.format_result(result))

──────────────────────────────────────────────────
LeBron James vs BOS
──────────────────────────────────────────────────
Points: 27.2 ± 6.8
Line: 25.5

  OVER  25.5:  59.9%  ███████████░░░░░░░░░
  UNDER 25.5:  40.1%  ████████░░░░░░░░░░░░

  ▸ Recommendation: OVER (moderate)
──────────────────────────────────────────────────


## Monte Carlo Simulation

Runs thousands of simulated games and counts how often the player goes over/under the line.

In [3]:
result = calc.run_monte_carlo_simulation(
    player_name='LeBron James',
    stat='pts',
    line=25.5,
    mean=27.2,
    std=6.8,
    opponent='BOS',
    date='2026-02-28',
    play_probability=0.95,   # 5% chance he sits
    num_sims=5000,
    team='LAL',
    is_home=False,
    base_mean=27.2,
    adjustments=[('Away Game', -0.3), ('Hot Streak', +0.5)]
)

print(calc.format_detailed_result(result))

══════════════════════════════════════════════════════════════════════
LeBron James (LAL) vs BOS — 2026-02-28
══════════════════════════════════════════════════════════════════════

┌─ PROJECTION CALCULATION ─────────────────────────────────────────────┐
│  Base Projection:      27.2 Points (from Monte Carlo sim)                 │
│  Away Game:           -0.3 Points                                 │
│  Hot Streak:           +0.5 Points                                │
│  ─────────────────────────────────────────────────────────────────── │
│  FINAL PROJECTION:     27.2 ± 6.8 Points                            │
│                                                                      │
│  95% Confidence: 13.9 - 40.6 Points                               │
│  Data Source: Monte Carlo (5000 simulations)                          │
└──────────────────────────────────────────────────────────────────────┘

┌─ OVER/UNDER: 25.5 POINTS ─────────────────────────────────────────┐
│                    

## Compare Multiple Lines

See which line offers the best edge.

In [4]:
print(calc.compare_lines(
    player_name='LeBron James',
    stat='pts',
    lines=[20.5, 22.5, 25.5, 27.5, 30.5, 32.5],
    mean=27.2,
    std=6.8,
    opponent='BOS'
))

────────────────────────────────────────────────────────────
LeBron James - Points Line Comparison
Projection: 27.2 ± 6.8
────────────────────────────────────────────────────────────
    Line       OVER      UNDER       Recommendation
────────────────────────────────────────────────────────────
    20.5      83.8%      16.2%        OVER (strong)
    22.5      75.5%      24.5%        OVER (strong)
    25.5      59.9%      40.1%      OVER (moderate)
    27.5      48.2%      51.8%     PASS (too close)
    30.5      31.4%      68.6%       UNDER (strong)
    32.5      21.8%      78.2%       UNDER (strong)
────────────────────────────────────────────────────────────


## Multi-Player Comparison

Compare over/under edges across players on the same slate.

In [5]:
players = [
    {'name': 'LeBron James',  'team': 'LAL', 'opp': 'BOS', 'stat': 'pts', 'line': 25.5, 'mean': 27.2, 'std': 6.8},
    {'name': 'Jayson Tatum',  'team': 'BOS', 'opp': 'LAL', 'stat': 'pts', 'line': 27.5, 'mean': 26.8, 'std': 7.1},
    {'name': 'Nikola Jokic',  'team': 'DEN', 'opp': 'GSW', 'stat': 'reb', 'line': 12.5, 'mean': 13.1, 'std': 3.2},
    {'name': 'Stephen Curry', 'team': 'GSW', 'opp': 'DEN', 'stat': 'pts', 'line': 28.5, 'mean': 26.4, 'std': 7.5},
    {'name': 'Luka Doncic',   'team': 'DAL', 'opp': 'MIL', 'stat': 'ast', 'line': 8.5,  'mean': 9.2,  'std': 2.8},
]

print(f"{'Player':<20} {'Stat':<6} {'Line':>6} {'Mean':>6} {'OVER':>8} {'UNDER':>8} {'Rec':>20}")
print('=' * 80)

for p in players:
    r = calc.calculate_from_projection(
        player_name=p['name'], stat=p['stat'], line=p['line'],
        mean=p['mean'], std=p['std'], opponent=p['opp']
    )
    print(f"{p['name']:<20} {p['stat']:<6} {p['line']:>6.1f} {p['mean']:>6.1f} "
          f"{r.prob_over*100:>7.1f}% {r.prob_under*100:>7.1f}% {r.recommendation:>20}")

Player               Stat     Line   Mean     OVER    UNDER                  Rec
LeBron James         pts      25.5   27.2    59.9%    40.1%      OVER (moderate)
Jayson Tatum         pts      27.5   26.8    46.1%    53.9%     PASS (too close)
Nikola Jokic         reb      12.5   13.1    57.4%    42.6%      OVER (moderate)
Stephen Curry        pts      28.5   26.4    39.0%    61.0%     UNDER (moderate)
Luka Doncic          ast       8.5    9.2    59.9%    40.1%      OVER (moderate)


## Injury-Adjusted Query

When a player is questionable, set `play_probability < 1.0` to factor in the DNP risk.

In [6]:
# Questionable player — 70% chance to play
result = calc.run_monte_carlo_simulation(
    player_name='Anthony Davis',
    stat='reb',
    line=10.5,
    mean=11.8,
    std=3.5,
    opponent='PHX',
    play_probability=0.70,
    num_sims=5000,
    team='LAL',
    is_home=True,
    base_mean=11.8,
    adjustments=[('Home Court', +0.5), ('Questionable - Knee', 0)]
)

print(calc.format_detailed_result(result))

══════════════════════════════════════════════════════════════════════
Anthony Davis (LAL) vs PHX
══════════════════════════════════════════════════════════════════════

┌─ PROJECTION CALCULATION ─────────────────────────────────────────────┐
│  Base Projection:      11.8 Rebounds (from Monte Carlo sim)                 │
│  Home Court:           +0.5 Rebounds                              │
│  Questionable - Knee:           +0.0 Rebounds                     │
│  ─────────────────────────────────────────────────────────────────── │
│  FINAL PROJECTION:     11.8 ± 3.5 Rebounds                            │
│                                                                      │
│  95% Confidence: 5.1 - 18.7 Rebounds                               │
│  Data Source: Monte Carlo (5000 simulations)                          │
└──────────────────────────────────────────────────────────────────────┘

┌─ OVER/UNDER: 10.5 REBOUNDS ─────────────────────────────────────────┐
│                         

## From Simulation Results (if available)

If you've run `python simulate_season.py --today`, this cell loads the projections from disk.

In [7]:
from src.query.projection_loader import ProjectionLoader

loader = ProjectionLoader(data_dir='data/sim_results')
df = loader.load_projections()

if df.empty:
    print('No simulation results found in data/sim_results/')
    print('Run:  python simulate_season.py --today')
else:
    info = loader.get_cache_info()
    print(f"Loaded {info['num_projections']} projections")
    print(f"Players: {', '.join(loader.get_available_players()[:10])}...")
    print(f"Teams:   {', '.join(loader.get_available_teams())}")

    # Example: query loaded projection
    # proj = loader.find_player('LeBron James')
    # if proj:
    #     r = calc.calculate_from_projection(
    #         player_name=proj.player_name, stat='pts', line=25.5,
    #         mean=proj.get_stat_mean('pts'), std=proj.get_stat_std('pts'),
    #         opponent=proj.opponent
    #     )
    #     print(calc.format_result(r))

Data directory not found: data/sim_results
No projection file found, returning empty DataFrame


No simulation results found in data/sim_results/
Run:  python simulate_season.py --today
